In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install BERTopic

In [3]:
!pip install bertopic
!pip install --upgrade joblib==1.1.0 # HDBScan fails to load with joblib==1.2.0
!pip install --upgrade git+https://github.com/scikit-learn-contrib/hdbscan.git#egg=hdbscan

  Using cached joblib-1.1.0-py2.py3-none-any.whl.metadata (5.2 kB)
Using cached joblib-1.1.0-py2.py3-none-any.whl (306 kB)
  Attempting uninstall: joblib
    Found existing installation: joblib 1.4.2
    Uninstalling joblib-1.4.2:
      Successfully uninstalled joblib-1.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
imbalanced-learn 0.12.4 requires joblib>=1.1.1, but you have joblib 1.1.0 which is incompatible.
scikit-learn 1.5.2 requires joblib>=1.2.0, but you have joblib 1.1.0 which is incompatible.
  Cloning https://github.com/scikit-learn-contrib/hdbscan.git to /tmp/pip-install-q5te_7hy/hdbscan_69b21eea27eb4512a34ae2e2b270019e
  Running command git clone --filter=blob:none --quiet https://github.com/scikit-learn-contrib/hdbscan.git /tmp/pip-install-q5te_7hy/hdbscan_69b21eea27eb4512a34ae2e2b270019e
  Resolved https://github.com/scikit-learn-contrib/hdbs

In [4]:

import os
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP # In case we want to make results deterministic and reproducible

In [5]:
with open("drive/MyDrive/stopwords.txt", 'r', encoding="utf8") as f:
    stop = f.read().splitlines()

vectorizer_model = CountVectorizer(ngram_range=(1, 1), stop_words=frozenset(stop))

In [6]:
# Prepare data
df = pd.read_csv('drive/MyDrive/preprocessed_lenta_titles.csv', encoding="utf-8")
df['text_lemm'].dropna(inplace=True)
docs = df['text_ready'].astype(str).values.tolist()


In [7]:
df.head()

,Unnamed: 0,title,tags,text_lower,text_punct,text_stop,text_common,text_rare,text_nonum,text_token,text_lemm,text_ready
0,0,В Курской области объявили опасность атаки БПЛА,Происшествия,в курской области объявили опасность атаки бпла,в курской области объявили опасность атаки бпла,курской области объявили опасность атаки бпла,курской объявили опасность атаки бпла,курской объявили опасность атаки бпла,курской объявили опасность атаки бпла,"['курской', 'объявили', 'опасность', 'атаки', ...",курский объявить опасность атака бпнуть,курский область объявить опасность атака бпнуть
1,1,Юристы прокомментировали иск пострадавших в от...,Следствие и суд,юристы прокомментировали иск пострадавших в от...,юристы прокомментировали иск пострадавших в от...,юристы прокомментировали иск пострадавших отно...,юристы прокомментировали иск пострадавших отно...,юристы прокомментировали иск пострадавших отно...,юристы прокомментировали иск пострадавших отно...,"['юристы', 'прокомментировали', 'иск', 'постра...",юрист прокомментировать иск пострадать отношен...,юрист прокомментировать иск пострадать отношен...
2,2,На Западе рассказали о нехватке Patriot для пе...,Политика,на западе рассказали о нехватке patriot для пе...,на западе рассказали о нехватке patriot для пе...,западе рассказали нехватке patriot передачи всу,западе нехватке patriot передачи,западе нехватке patriot передачи,западе нехватке patriot передачи,"['западе', 'нехватке', 'patriot', 'передачи']",запад нехватка patriot передача,запад рассказать нехватка patriot передача всу
3,3,Очередные российские спортсмены отказались от ...,Летние виды,очередные российские спортсмены отказались от ...,очередные российские спортсмены отказались от ...,очередные российские спортсмены отказались уча...,очередные спортсмены отказались участия олимпиаде,очередные спортсмены отказались участия олимпиаде,очередные спортсмены отказались участия олимпиаде,"['очередные', 'спортсмены', 'отказались', 'уча...",очередной спортсмен отказаться участие олимпиада,очередной российский спортсмен отказаться учас...
4,4,Байден и Си Цзиньпин обсудили ситуацию вокруг ...,Политика,байден и си цзиньпин обсудили ситуацию вокруг ...,байден и си цзиньпин обсудили ситуацию вокруг ...,байден си цзиньпин обсудили ситуацию вокруг ti...,байден си цзиньпин обсудили ситуацию вокруг ti...,байден си цзиньпин обсудили ситуацию вокруг ti...,байден си цзиньпин обсудили ситуацию вокруг ti...,"['байден', 'си', 'цзиньпин', 'обсудили', 'ситу...",байден си цзиньпин обсудить ситуация вокруг ti...,байден си цзиньпин обсудить ситуация вокруг ti...


In [8]:
from bertopic import BERTopic
import pandas as pd
import nltk
nltk.download('stopwords')
#from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
df.head(16616)

,Unnamed: 0,title,tags,text_lower,text_punct,text_stop,text_common,text_rare,text_nonum,text_token,text_lemm,text_ready
0,0,В Курской области объявили опасность атаки БПЛА,Происшествия,в курской области объявили опасность атаки бпла,в курской области объявили опасность атаки бпла,курской области объявили опасность атаки бпла,курской объявили опасность атаки бпла,курской объявили опасность атаки бпла,курской объявили опасность атаки бпла,"['курской', 'объявили', 'опасность', 'атаки', ...",курский объявить опасность атака бпнуть,курский область объявить опасность атака бпнуть
1,1,Юристы прокомментировали иск пострадавших в от...,Следствие и суд,юристы прокомментировали иск пострадавших в от...,юристы прокомментировали иск пострадавших в от...,юристы прокомментировали иск пострадавших отно...,юристы прокомментировали иск пострадавших отно...,юристы прокомментировали иск пострадавших отно...,юристы прокомментировали иск пострадавших отно...,"['юристы', 'прокомментировали', 'иск', 'постра...",юрист прокомментировать иск пострадать отношен...,юрист прокомментировать иск пострадать отношен...
2,2,На Западе рассказали о нехватке Patriot для пе...,Политика,на западе рассказали о нехватке patriot для пе...,на западе рассказали о нехватке patriot для пе...,западе рассказали нехватке patriot передачи всу,западе нехватке patriot передачи,западе нехватке patriot передачи,западе нехватке patriot передачи,"['западе', 'нехватке', 'patriot', 'передачи']",запад нехватка patriot передача,запад рассказать нехватка patriot передача всу
3,3,Очередные российские спортсмены отказались от ...,Летние виды,очередные российские спортсмены отказались от ...,очередные российские спортсмены отказались от ...,очередные российские спортсмены отказались уча...,очередные спортсмены отказались участия олимпиаде,очередные спортсмены отказались участия олимпиаде,очередные спортсмены отказались участия олимпиаде,"['очередные', 'спортсмены', 'отказались', 'уча...",очередной спортсмен отказаться участие олимпиада,очередной российский спортсмен отказаться учас...
4,4,Байден и Си Цзиньпин обсудили ситуацию вокруг ...,Политика,байден и си цзиньпин обсудили ситуацию вокруг ...,байден и си цзиньпин обсудили ситуацию вокруг ...,байден си цзиньпин обсудили ситуацию вокруг ti...,байден си цзиньпин обсудили ситуацию вокруг ti...,байден си цзиньпин обсудили ситуацию вокруг ti...,байден си цзиньпин обсудили ситуацию вокруг ti...,"['байден', 'си', 'цзиньпин', 'обсудили', 'ситу...",байден си цзиньпин обсудить ситуация вокруг ti...,байден си цзиньпин обсудить ситуация вокруг ti...
...,...,...,...,...,...,...,...,...,...,...,...,...
16611,16611,Безруков пожаловался на говоривших голосом Хаб...,Кино,безруков пожаловался на говоривших голосом хаб...,безруков пожаловался на говоривших голосом хаб...,безруков пожаловался говоривших голосом хабенс...,безруков пожаловался говоривших голосом хабенс...,безруков пожаловался говоривших голосом хабенс...,безруков пожаловался говоривших голосом хабенс...,"['безруков', 'пожаловался', 'говоривших', 'гол...",безруков пожаловаться говорить голос хабенский...,безруков пожаловаться говорить голос хабенский...
16612,16612,Генсек НАТО поздравил Швецию с ратификацией ее...,Политика,генсек нато поздравил швецию с ратификацией ее...,генсек нато поздравил швецию с ратификацией ее...,генсек нато поздравил швецию ратификацией заяв...,генсек поздравил швецию ратификацией заявки ве...,генсек поздравил швецию ратификацией заявки ве...,генсек поздравил швецию ратификацией заявки ве...,"['генсек', 'поздравил', 'швецию', 'ратификацие...",генсек поздравить швеция ратификация заявка ве...,генсек нато поздравить швеция ратификация заяв...
16613,16613,Детей завалило песком во время игры на пляже,Люди,детей завалило песком во время игры на пляже,детей завалило песком во время игры на пляже,детей завалило песком время игры пляже,детей завалило песком время игры пляже,детей завалило песком время игры пляже,детей завалило песком время игры пляже,"['детей', 'завалило', 'пе

In [10]:
docs

['курский область объявить опасность атака бпнуть',
 'юрист прокомментировать иск пострадать отношение владелец крокус',
 'запад рассказать нехватка patriot передача всу',
 'очередной российский спортсмен отказаться участие олимпиада 2024',
 'байден си цзиньпин обсудить ситуация вокруг tiktok',
 'последствие мощный землетрясение тайвань показать видео',
 'родственник погибнуть крокус назначить 66 пенсия',
 'цунами достигнуть японский остров землетрясение близ тайвань',
 'украина заявить ужесточение мобилизация из за военный закон',
 'британия осудить северный корея из за пуск баллистический ракета',
 'ростов выйти следующий этап кубок россия',
 'локомотив повести серия трактор полуфинал кубок гагарин',
 'назвать число россиянин кредит',
 'запад создать план сохранение поставка оружие украина возвращение трамп',
 'близ тайвань зафиксировать четыре землетрясение',
 'диетолог раскрыть неожиданный польза апельсин',
 'украинский миллиардер стать бедный',
 'крупный производитель чип эвакуиро

In [11]:
from nltk.corpus import stopwords


In [12]:
# Remove stopwords **after** generating the clusters
# NOTE: You can also perform the lemmatization here
my_stopwords = stopwords.words("russian")
vectorizer_model = CountVectorizer(stop_words=my_stopwords)

In [13]:
#from hdbscan import HDBSCAN
documents = df['text_ready'].astype(str).values.tolist()
umap_model = UMAP(n_neighbors=10, n_components=3,
                  min_dist=0.0, metric='cosine', random_state=13)

In [14]:


# Load KBLab's Swedish sentence transformer model
sentence_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2", device="cuda")

# Initialize BERTopic with the settings we want
topic_model = BERTopic(embedding_model=sentence_model,
                       vectorizer_model=vectorizer_model,
                       umap_model=umap_model,
                       calculate_probabilities=True,
                       verbose=True)

topic_model.save("drive/MyDrive/lenta_model", serialization="pickle")
# Fit the model
#topics, probs = topic_model.fit_transform(documents)

2024-12-07 09:03:20,784 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


In [15]:
# Fit the model
topics, probs = topic_model.fit_transform(documents)

2024-12-07 09:03:28,233 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/2157 [00:00<?, ?it/s]

2024-12-07 09:04:53,923 - BERTopic - Embedding - Completed ✓
2024-12-07 09:04:53,925 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-12-07 09:06:28,406 - BERTopic - Dimensionality - Completed ✓
2024-12-07 09:06:28,408 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-12-07 10:32:14,768 - BERTopic - Cluster - Completed ✓
2024-12-07 10:32:14,794 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-12-07 10:32:16,286 - BERTopic - Representation - Completed ✓


In [16]:
while True:
    pass

KeyboardInterrupt: 

In [17]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,25003,-1_москва_город_всу_дело,"[москва, город, всу, дело, сво, попасть, видео...",[военный эксперт назвать возможный срок операц...
1,0,1276,0_конфликт_украина_patriot_запад,"[конфликт, украина, patriot, запад, завершение...",[захаров предупредить сша риска из за поставка...
2,1,516,1_полицейский_суд_арестовать_судья,"[полицейский, суд, арестовать, судья, арест, в...","[мужчина нож напасть российский полицейский, с..."
3,2,470,2_авиакомпания_рейс_самолёт_аэропорт,"[авиакомпания, рейс, самолёт, аэропорт, полёт,...",[пять международный авиакомпания отменить рейс...
4,3,390,3_зеленский_залужный_офис_формула,"[зеленский, залужный, офис, формула, встреча, ...",[залужный прокомментировать разговор зеленский...
...,...,...,...,...,...
813,812,10,812_жертва_550_папуа_избирком,"[жертва, 550, папуа, избирком, увеличиться, гв...","[увеличиться количество жертва теракт крокус, ..."
814,813,10,813_полковник_тактика_генерал_супероружие,"[полковник, тактика, генерал, супероружие, сур...",[генерал всу заявить смена тактика российский ...
815,814,10,814_ядерный_арсенал_китай_боеголовка,"[ядерный, арсенал, китай, боеголовка, порог, м...",[сша допустить отставание китай число ядерный ...
816,815,10,815_левша_барс_шлем_ка,"[левша, барс, шлем, ка, промышленность, натовс...",[ран сотрудничать национальный академия четыре...


In [18]:
topic_model.get_topic(0)

[('конфликт', 0.015172846947149448),
 ('украина', 0.009338568722134228),
 ('patriot', 0.008944376194611319),
 ('запад', 0.006034054854434247),
 ('завершение', 0.005879901251972224),
 ('украинец', 0.005830997015733302),
 ('киев', 0.005754700463025255),
 ('вступление', 0.005530602892475148),
 ('потеря', 0.005496815910868003),
 ('нато', 0.0048575615518353455)]

In [19]:
topic_model.get_representative_docs(0)

['захаров предупредить сша риска из за поставка украина система пво patriot',
 'трамп назвать условие завершение конфликт украина',
 'сша назвать условие поставка patriot украина']

In [20]:
topic_model.get_topic(4) # topic_model.get_topic(similar_topics[0])

[('беспилотник', 0.050484605896433145),
 ('дрон', 0.04648640926922218),
 ('сбить', 0.020986202506033034),
 ('брянский', 0.01995338025065552),
 ('область', 0.01729538323193954),
 ('атаковать', 0.01688745432565012),
 ('белгородский', 0.01621614770478666),
 ('атака', 0.014997914216955994),
 ('нефтебаза', 0.01409358927974908),
 ('курский', 0.013388103188178685)]

In [23]:
topic_model.visualize_topics()

In [21]:
fig = topic_model.visualize_topics()
fig.write_html("/content/sample_data/lenta_topics.html")

In [22]:
topic_model.visualize_hierarchy()

In [24]:
hierarchical_topics = topic_model.hierarchical_topics(documents)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

100%|██████████| 816/816 [00:05<00:00, 162.01it/s]


In [25]:
topic_model.reduce_topics(docs, nr_topics=20)


2024-12-07 10:34:13,520 - BERTopic - Topic reduction - Reducing number of topics
2024-12-07 10:34:14,357 - BERTopic - Topic reduction - Reduced number of topics from 818 to 20


In [26]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,25003,-1_россия_российский_украина_назвать,"[россия, российский, украина, назвать, россиян...","[сша заявить подготовка польша война россия, с..."
1,0,26122,0_украина_россия_российский_россиянин,"[украина, россия, российский, россиянин, заяви...",[глава граничить украина российский регион соо...
2,1,8449,1_израиль_всу_сша_назвать,"[израиль, всу, сша, назвать, заявить, жильё, а...",[британия призвать напасть иран случай удар из...
3,2,3543,2_летний_назвать_способ_женщина,"[летний, назвать, способ, женщина, врач, показ...",[51 летний модель показать фигура красный купа...
4,3,1084,3_журналист_блогерш_блогер_youtube,"[журналист, блогерш, блогер, youtube, twitter,...","[российский журналист попасть обстрел донбасс,..."
5,4,1042,4_китай_корея_южный_таиланд,"[китай, корея, южный, таиланд, китайский, сша,...","[назвать перспектива отношение тайвань китай, ..."
6,5,879,5_коронавирус_covid_19_рак,"[коронавирус, covid, 19, рак, омикрон, грипп, ...",[россия выявить десятка случай заражение новый...
7,6,851,6_медведев_медведь_обезьяна_оспа,"[медведев, медведь, обезьяна, оспа, акула, роб...",[медведев прокомментировать условие британия п...
8,7,582,7_ипотека_доллар_ставка_криптовалюта,"[ипотека, доллар, ставка, криптовалюта, пенсия...","[госдума назвать льготный ипотека ошибка, сбер..."
9,8,405,8_apple_iphone_смартфон_samsung,"[apple, iphone, смартфон, samsung, android, ск...","[apple увеличить производство iphone 14, apple..."


In [27]:
topic_model.get_topic(3)

[('журналист', 0.10100323448989688),
 ('блогерш', 0.07869922968372589),
 ('блогер', 0.07636527018065685),
 ('youtube', 0.04467010976619872),
 ('twitter', 0.03996798126247909),
 ('google', 0.03508179184434601),
 ('популярный', 0.03269055204600515),
 ('российский', 0.031578384142788504),
 ('журналистка', 0.030707751586051282),
 ('сбер', 0.030168903417721125)]

In [28]:
topic_model.visualize_topics()


In [32]:
fig4 = topic_model.visualize_barchart(width=280, height=330,top_n_topics=20, n_words=20)
topic_model.visualize_barchart(width=280, height=330,top_n_topics=20, n_words=20)

In [35]:
fig4.write_html("/content/drive/MyDrive/lenta_topic_20wordBars.html")

In [37]:
fig5 = topic_model.visualize_heatmap(n_clusters=9)
topic_model.visualize_heatmap(n_clusters=9)
fig5.write_html("/content/sample_data/lenta_topic_20corr.html")